# 03 · Recurrence judge evaluation on 300 hand-labelled SDRS pairs

**Task.** Given two snag narratives from the same aircraft family, decide whether they describe the *same underlying defect signature*: the failing system, the symptom and the operating condition align, even when the wording differs.

**Set.** 300 pairs in three strata built by `services/daleel/eval.py build`: 100 nearest-neighbour pairs (same chapter, same family, cosine ≥ 0.6), 100 random same-chapter pairs, 100 cross-chapter pairs. Labelled by reading every pair (labeller recorded in the CSV). 60 pairs form a dev split used only to tune thresholds; all numbers below are on the remaining 240.

**Methods.** TF-IDF cosine (word 1–2 grams), all-MiniLM-L6-v2 cosine, the production `EmbeddingJudge` (fixed threshold), and `ClaudeJudge` (claude-sonnet-4-6) when an API key is configured.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
m = json.load(open('../data/metrics/daleel_recurrence.json'))
labels = pd.read_csv('../data/labels/recurrence_pairs.csv')
print(f"pairs {m['n_pairs']} · dev {m['n_dev']} · test {m['n_test']} · positives in test {m['positives_test']}")
print('labeller:', m['labeller'])
labels.groupby('stratum').label.agg(pairs='count', positives='sum')

pairs 300 · dev 60 · test 240 · positives in test 83
labeller: Claude Fable 5.1 (hand-labelled by reading each pair, 2026-09-09); rubric in docs/EVALUATION.md


,pairs,positives
stratum,,
chapter,100,21
cross,100,0
nn,100,85


In [2]:
rows = []
for k, v in m['models'].items():
    rows.append({'method': k, 'threshold': v.get('threshold', m['recommended_embedding_threshold'] if k=='embedding_judge' else None), 'precision': v['precision'], 'recall': v['recall'], 'f1': v['f1'], 'model_version': v.get('model_version','')})
table = pd.DataFrame(rows).set_index('method').round(3)
table

,threshold,precision,recall,f1,model_version
method,,,,,
tfidf_cosine,0.100,0.762,0.928,0.837,
embedding_cosine,0.590,0.802,0.928,0.860,
embedding_judge,0.775,0.910,0.735,0.813,embedding-judge/all-MiniLM-L6-v2@cos>=0.775


## Where each method fails

Per-stratum results for the production judge. Nearest-neighbour pairs are where recurrence detection actually happens (retrieval already narrowed to k=20); the same-chapter stratum holds the paraphrase-heavy positives that a fixed cosine threshold under-recalls, which is the case the LLM judge exists for.

In [3]:
per = {st: {k: v for k, v in d.items() if v} for st, d in m['per_stratum'].items()}
pd.DataFrame({(st, k): v for st, d in per.items() for k, v in d.items()}).T.round(3)

,,precision,recall,f1,n
nn,embedding_judge,0.909,0.909,0.909,80.0
chapter,embedding_judge,1.000,0.059,0.111,78.0
cross,embedding_judge,0.000,0.000,0.000,82.0


In [4]:
sw = pd.DataFrame([{'threshold': s['threshold'], 'dev_precision': s['dev']['precision'], 'dev_recall': s['dev']['recall'], 'test_precision': s['test']['precision'], 'test_recall': s['test']['recall'], 'test_f1': s['test']['f1']} for s in m['embedding_judge_sweep']])
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(sw.threshold, sw.test_precision, label='precision (test)', color='#4E8C6A')
ax.plot(sw.threshold, sw.test_recall, label='recall (test)', color='#E8A317')
ax.plot(sw.threshold, sw.test_f1, label='F1 (test)', color='#8FA8B5', ls='--')
rec = m['recommended_embedding_threshold']
ax.axvline(rec, color='#C4342B', lw=1); ax.text(rec+0.005, 0.2, f'production threshold {rec}', color='#C4342B')
ax.set_xlabel('cosine threshold'); ax.set_ylim(0,1.02); ax.legend(loc='lower left'); ax.set_title('Embedding judge operating points')
fig.tight_layout(); fig.savefig('../docs/figures/eval_recurrence_sweep.png'); plt.close(fig)
print('recommended threshold', rec, '->', {k: round(v,3) for k, v in m['recommended_embedding_threshold_test'].items() if k!='n'})

recommended threshold 0.775 -> {'precision': 0.91, 'recall': 0.735, 'f1': 0.813}


## Reading the numbers

* Cosine similarity, TF-IDF or embedding, reaches F1 ≈ 0.85 when its threshold is tuned on labelled pairs, but the tuned thresholds (0.10 for TF-IDF, 0.59 for embeddings) are so permissive that in production, where every candidate is compared against 20 same-chapter neighbours, they would merge most of a chapter into one signature.
* The production threshold is chosen for precision first: the highest recall with dev precision ≥ 0.90. On test that is precision 0.91 and recall 0.73. A false recurrence is a false alert to a certifying engineer; a missed one is caught on the next report.
* The recall gap sits in the same-chapter stratum, where positives are true paraphrases. That is the case the LLM judge is designed to cover; its numbers appear in this table when `ANTHROPIC_API_KEY` is set and the evaluation is re-run.